# FlowMatching Architecture Search for ANCOVA NPE

Multi-objective Bayesian optimization (Optuna) to find the best FlowMatching
architecture for 2-arm ANCOVA amortized posterior estimation.

**Objectives:** minimize calibration error + minimize parameter count (Pareto front).

**Output:** best `ANCOVAConfig` as JSON for the deployment script + Optuna study DB.

In [ ]:
import os

if not os.environ.get("KERAS_BACKEND"):
    os.environ["KERAS_BACKEND"] = "torch"

from pathlib import Path

import numpy as np
import bayesflow_hpo as hpo
import matplotlib.pyplot as plt

from bayesflow_rct.models.ancova.model import (
    ANCOVAConfig,
    create_simulator,
    create_ancova_adapter,
    create_validation_grid,
    hpo_params_to_config,
)
from bayesflow_rct.models.ancova.validation import build_validation_dataset

np.set_printoptions(suppress=True)
RNG = np.random.default_rng(2025)

# Simulation defaults (prior ranges, N range) and training hyperparameters
# (epochs, batch_size, early stopping). Architecture fields will be
# overridden by HPO — the config exported at the end is the final version.
config = ANCOVAConfig()
simulator = create_simulator(config, RNG)
adapter = create_ancova_adapter()

print(f"Config: inference={config.inference_network_type}, "
      f"epochs={config.epochs}, batch={config.batch_size}")
print(f"Simulator: {len(simulator.sample(1).keys())} output keys")

## Validation Dataset

Build an ANCOVA-specific validation dataset from a condition grid.
The dataset is cached to disk for reproducibility across runs.

In [ ]:
conditions = create_validation_grid(extended=True)

# 144 conditions × 500 sims = 72 000 total simulations
val_data_path = Path("data/ancova_hpo_validation")
if val_data_path.exists():
    val_data = hpo.load_validation_dataset(val_data_path)
    print(f"Loaded cached validation dataset ({len(val_data.simulations)} conditions)")
else:
    val_data = build_validation_dataset(conditions, n_sims=500, rng=RNG)
    hpo.save_validation_dataset(val_data, val_data_path)
    print(f"Generated and saved validation dataset ({len(conditions)} conditions)")

print(f"Grid: n_total × prior_df × prior_scale × b_covariate × b_arm_treat × p_alloc")

## Search Space

FlowMatching with optimal transport always on (`fm_use_ot=True`).
See `docs/superpowers/specs/2026-03-11-ancova-optimization-notebook-design.md`
for rationale behind custom ranges.

In [ ]:
search_space = hpo.CompositeSearchSpace(
    inference_space=hpo.FlowMatchingSpace(
        subnet_width=hpo.IntDimension("fm_subnet_width", 32, 256, step=32),
        subnet_depth=hpo.IntDimension("fm_subnet_depth", 1, 4),
        dropout=hpo.FloatDimension("fm_dropout", 0.0, 0.2),
        use_optimal_transport=hpo.CategoricalDimension("fm_use_ot", choices=[True]),
    ),
    summary_space=hpo.DeepSetSpace(
        summary_dim=hpo.IntDimension("ds_summary_dim", 4, 16),
        width=hpo.IntDimension("ds_width", 32, 128, step=16),
        depth=hpo.IntDimension("ds_depth", 1, 4),
        dropout=hpo.FloatDimension("ds_dropout", 0.05, 0.5),
    ),
    training_space=hpo.TrainingSpace(
        initial_lr=hpo.FloatDimension("initial_lr", 1e-5, 5e-3, log=True),
        batch_size=hpo.IntDimension("batch_size", 128, 832, step=32),
    ),
)

print(f"Search space: {len(search_space.inference_space.dimensions)} inference "
      f"+ {len(search_space.summary_space.dimensions)} summary "
      f"+ {len(search_space.training_space.dimensions)} training dimensions")

## Run Optimization

Single `hpo.optimize()` call handles study creation, objective wiring,
and the optimization loop.

> **Resume:** `resume=True` means re-running this cell continues from
> where it left off. Delete the SQLite DB to start fresh.

> **Known limitation:** `hpo.optimize()` does not expose early stopping
> params, so trials use `ObjectiveConfig` defaults (patience=5, window=7)
> instead of `ANCOVAConfig` values (patience=10, window=5).
> See [bayesflow-hpo#6](https://github.com/matthiaskloft/bayesflow-hpo/issues/6).

In [ ]:
N_TRIALS = 100  # Adjust based on compute budget

study = hpo.optimize(
    simulator=simulator,
    adapter=adapter,
    param_keys=["b_group"],
    data_keys=["outcome", "covariate", "group"],
    validation_data=val_data,
    search_space=search_space,
    inference_conditions=["N", "p_alloc", "prior_df", "prior_scale"],
    n_trials=N_TRIALS,
    epochs=config.epochs,
    batches_per_epoch=config.batches_per_epoch,
    cost_metric="param_count",
    storage="sqlite:///ancova_hpo.db",
    study_name="ancova_flowmatching",
    resume=True,
)

print(f"\nOptimization complete!")
print(f"Total trials: {len(study.trials)}")
print(f"Pareto-optimal trials: {len(study.best_trials)}")

In [ ]:
# To monitor optimization in real-time, launch the Optuna Dashboard:
#   pip install optuna-dashboard
#   optuna-dashboard sqlite:///ancova_hpo.db
print("Dashboard command:")
print("  optuna-dashboard sqlite:///ancova_hpo.db")

## Analyze Pareto Front

In [ ]:
# Text summary
print(hpo.summarize_study(study))

# Pareto-optimal trials as DataFrame
df = hpo.trials_to_dataframe(study)
pareto = hpo.get_pareto_trials(study)
pareto_df = df[df["trial_number"].isin({t.number for t in pareto})].copy()
pareto_df = pareto_df.sort_values("objective_0").reset_index(drop=True)
display(pareto_df)

print(f"\n{len(pareto)} Pareto-optimal configurations found")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plt.sca(axes[0])
hpo.plot_pareto_front(study)

plt.sca(axes[1])
hpo.plot_param_importance(study)

plt.tight_layout()
plt.show()

## Export Best Config

Save the best `ANCOVAConfig` as JSON for the deployment script.
Includes provenance metadata to trace back to the Optuna trial.

In [ ]:
from dataclasses import asdict
import json

# Select Pareto-best by calibration error (first objective)
best_trial = sorted(pareto, key=lambda t: t.values[0])[0]
config_best = hpo_params_to_config(best_trial.params)

export = {
    "config": asdict(config_best),
    "provenance": {
        "trial_number": best_trial.number,
        "objectives": list(best_trial.values),
        "study_name": "ancova_flowmatching",
        "study_db": "ancova_hpo.db",
    },
}

config_path = Path("configs/ancova_best.json")
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(export, indent=2, default=str))

print(f"Best trial: #{best_trial.number}")
print(f"  Calibration error: {best_trial.values[0]:.4f}")
print(f"  Param score:       {best_trial.values[1]:.4f}")
print(f"\nConfig saved to: {config_path}")
print(f"\nKey architecture:")
print(f"  FlowMatching: widths={config_best.inference_widths}, "
      f"dropout={config_best.inference_dropout:.2f}, "
      f"OT={config_best.inference_use_optimal_transport}")
print(f"  DeepSet: dim={config_best.summary_dim}, "
      f"depth={config_best.summary_depth}, "
      f"width={config_best.summary_width}, "
      f"dropout={config_best.summary_dropout:.2f}")
print(f"  Training: lr={config_best.initial_lr:.2e}, "
      f"batch={config_best.batch_size}")